# NB01 - Decouverte generale du depot KDK

Ce notebook explore le contenu du repertoire `kdk` copie dans `rag/data/kdk/`.

**Objectifs :**
- Scanner tous les fichiers et identifier les types (extensions)
- Determiner les repertoires et fichiers a exclure (node_modules, dist, .git, binaires, etc.)
- Etablir une vue d'ensemble du depot par grandes zones (`core`, `map`, `docs`, `test`, `vite`, `extras`)
- Identifier les familles de fichiers qui demanderont des strategies de chunking differentes
- Preparer les analyses specialisees, en particulier `docs/` qui sera approfondi dans `nb02`


In [23]:
import os
import re
from pathlib import Path
from collections import Counter, defaultdict

BASE_DIR = Path.cwd().resolve()
if not (BASE_DIR / "data" / "kdk").exists():
    BASE_DIR = BASE_DIR.parent

DATA_DIR = BASE_DIR / "data" / "kdk"
print(f"Repertoire de donnees : {DATA_DIR.resolve()}")


Repertoire de donnees : D:\kalisio\code_rag\data\kdk


## 1. Scan complet : decouverte de tous les fichiers

Premiere etape : scanner l'arborescence en excluant **uniquement les repertoires**
qui ne contiennent evidemment pas de contenu source (dependances, VCS, caches).

On liste ensuite **tous** les types de fichiers trouves pour decider lesquels exclure.


In [24]:
EXCLUDED_DIRS = {
    "node_modules",
    ".git",
    "dist",
    "build",
    ".output",
    "coverage",
    ".nyc_output",
    ".c8",
    "__pycache__",
    ".cache",
    ".github",
}

def scanner_fichiers(racine, excluded_dirs):
    """Parcourt l'arborescence en excluant certains repertoires."""
    fichiers = []
    for dirpath, dirnames, filenames in os.walk(racine):
        dirnames[:] = [d for d in dirnames if d not in excluded_dirs]
        for f in filenames:
            fichiers.append(Path(dirpath) / f)
    return fichiers

# Scan complet : seuls les repertoires techniques sont exclus
# afin d'etablir un inventaire le plus large possible.
tous_fichiers_bruts = scanner_fichiers(DATA_DIR, EXCLUDED_DIRS)
print(f"Fichiers trouves (apres exclusion des repertoires) : {len(tous_fichiers_bruts)}")

def formater_taille(octets):
    """Formate une taille en octets de maniere lisible."""
    for unite in ["o", "Ko", "Mo", "Go"]:
        if octets < 1024:
            return f"{octets:.1f} {unite}"
        octets /= 1024
    return f"{octets:.1f} To"

inventaire = {}
for f in tous_fichiers_bruts:
    ext = f.suffix.lower() if f.suffix else "(sans extension)"
    if ext not in inventaire:
        inventaire[ext] = {"nb": 0, "taille": 0}
    inventaire[ext]["nb"] += 1
    try:
        inventaire[ext]["taille"] += f.stat().st_size
    except OSError:
        pass

print(f"\n{'Extension':<20} {'Nombre':>8} {'Taille':>12}")
print("-" * 42)
for ext, data in sorted(inventaire.items(), key=lambda x: x[1]["taille"], reverse=True):
    print(f"{ext:<20} {data['nb']:>8} {formater_taille(data['taille']):>12}")

taille_brute = sum(d["taille"] for d in inventaire.values())
print(f"\nTotal : {len(tous_fichiers_bruts)} fichiers, {formater_taille(taille_brute)}")
print(f"Extensions distinctes : {len(inventaire)}")



Fichiers trouves (apres exclusion des repertoires) : 970

Extension              Nombre       Taille
------------------------------------------
.png                       71      16.7 Mo
.json                      26       3.5 Mo
.svg                       40       3.5 Mo
.xml                       16       2.7 Mo
.js                       362       1.9 Mo
.lock                       3       1.1 Mo
.dods                       3       1.0 Mo
.vue                      242     832.5 Ko
.md                       142     558.5 Ko
.gif                        1     373.6 Ko
.drawio                     8     323.8 Ko
.txt                        1      71.3 Ko
.mjs                       18      55.4 Ko
.tif                        1      39.4 Ko
.cjs                        6      13.2 Ko
.ejs                        9       4.3 Ko
.sh                         4       4.2 Ko
.html                       2       3.7 Ko
.scss                       2       2.3 Ko
(sans extension)            8       2.2

## 2. Strategie d'exclusion basee sur les decouvertes

A partir de l'inventaire ci-dessus, on identifie :
- **Fichiers specifiques** sans valeur semantique (lock files, changelogs auto-generes)
- **Extensions** dont le contenu n'est pas exploitable pour le RAG (images, binaires, etc.)


In [25]:
EXCLUDED_FILES = {
    "yarn.lock",
    "CHANGELOG.md",
}

EXCLUDED_EXTENSIONS = {
    ".png",
    ".gif",
    ".tif",
    ".svg",
    ".lock",
}

tous_fichiers = [
    f for f in tous_fichiers_bruts
    if f.name not in EXCLUDED_FILES
    and f.suffix.lower() not in EXCLUDED_EXTENSIONS
]

nb_exclus = len(tous_fichiers_bruts) - len(tous_fichiers)
taille_retenue = sum(f.stat().st_size for f in tous_fichiers)
taille_exclue = taille_brute - taille_retenue

print(f"Fichiers avant exclusions : {len(tous_fichiers_bruts)}")
print(f"Fichiers exclus           : {nb_exclus}")
print(f"Fichiers retenus          : {len(tous_fichiers)}")
print(f"Taille exclue             : {formater_taille(taille_exclue)}")

print("\nDetail des exclusions :")
for f_name in sorted(EXCLUDED_FILES):
    matches = [f for f in tous_fichiers_bruts if f.name == f_name]
    if matches:
        t = sum(f.stat().st_size for f in matches)
        print(f"  {f_name:<25} {len(matches):>3} fichier(s)  {formater_taille(t):>10}")
for ext in sorted(EXCLUDED_EXTENSIONS):
    matches = [f for f in tous_fichiers_bruts if f.suffix.lower() == ext]
    if matches:
        t = sum(f.stat().st_size for f in matches)
        print(f"  *{ext:<24} {len(matches):>3} fichier(s)  {formater_taille(t):>10}")



Fichiers avant exclusions : 970
Fichiers exclus           : 117
Fichiers retenus          : 853
Taille exclue             : 21.8 Mo

Detail des exclusions :
  CHANGELOG.md                1 fichier(s)     55.2 Ko
  yarn.lock                   3 fichier(s)      1.1 Mo
  *.gif                       1 fichier(s)    373.6 Ko
  *.lock                      3 fichier(s)      1.1 Mo
  *.png                      71 fichier(s)     16.7 Mo
  *.svg                      40 fichier(s)      3.5 Mo
  *.tif                       1 fichier(s)     39.4 Ko


## 3. Repartition et taille des fichiers retenus

Cette vue permet d'identifier les formats dominants une fois le bruit evident retire.


In [26]:
compteur_ext = Counter()
taille_par_ext = Counter()
for f in tous_fichiers:
    ext = f.suffix.lower() if f.suffix else "(sans extension)"
    compteur_ext[ext] += 1
    try:
        taille_par_ext[ext] += f.stat().st_size
    except OSError:
        pass

print(f"{'Extension':<20} {'Nombre':>8} {'Taille':>12}")
print("-" * 42)
for ext, taille in taille_par_ext.most_common():
    print(f"{ext:<20} {compteur_ext[ext]:>8} {formater_taille(taille):>12}")

taille_totale = sum(taille_par_ext.values())
print(f"\nTotal : {len(tous_fichiers)} fichiers, {formater_taille(taille_totale)}")
print(f"Extensions distinctes : {len(compteur_ext)}")



Extension              Nombre       Taille
------------------------------------------
.json                      26       3.5 Mo
.xml                       16       2.7 Mo
.js                       362       1.9 Mo
.dods                       3       1.0 Mo
.vue                      242     832.5 Ko
.md                       141     503.3 Ko
.drawio                     8     323.8 Ko
.txt                        1      71.3 Ko
.mjs                       18      55.4 Ko
.cjs                        6      13.2 Ko
.ejs                        9       4.3 Ko
.sh                         4       4.2 Ko
.html                       2       3.7 Ko
.scss                       2       2.3 Ko
(sans extension)            8       2.2 Ko
.das                        1       1.9 Ko
.skip                       1     1015.0 o
.dds                        1      608.0 o
.properties                 1      206.0 o
.css                        1      105.0 o

Total : 853 fichiers, 10.9 Mo
Extensions distinctes :

## 4. Vue d'ensemble du depot par repertoires

On ne veut pas seulement connaitre les extensions : il faut aussi savoir **ou** se trouve la masse utile.
Cette vue sert a reperer les zones du depot qui devront etre traitees differemment dans le pipeline RAG.


In [27]:
par_zone = defaultdict(list)
for f in tous_fichiers:
    rel = f.relative_to(DATA_DIR)
    zone = rel.parts[0] if len(rel.parts) > 1 else "(racine)"
    par_zone[zone].append(f)

print(f"{'Zone':<12} {'Fichiers':>8} {'Taille':>12} {'Ext dominantes'}")
print("-" * 80)
for zone, fichiers in sorted(par_zone.items(), key=lambda item: len(item[1]), reverse=True):
    taille = sum(f.stat().st_size for f in fichiers)
    ext_counter = Counter(f.suffix.lower() if f.suffix else "(sans extension)" for f in fichiers)
    top_ext = ", ".join(f"{ext}:{nb}" for ext, nb in ext_counter.most_common(4))
    print(f"{zone:<12} {len(fichiers):>8} {formater_taille(taille):>12} {top_ext}")

print("\nFichiers a la racine :")
for f in sorted(par_zone["(racine)"]):
    print(f"  {f.name}")



Zone         Fichiers       Taille Ext dominantes
--------------------------------------------------------------------------------
core              305     918.8 Ko .vue:150, .js:149, .json:6
map               238       1.3 Mo .js:142, .vue:87, .json:6, .cjs:3
docs              167       3.5 Mo .md:139, .xml:15, .drawio:8, .json:2
test               50       4.6 Mo .js:21, .json:10, .ejs:9, .dods:3
extras             37     579.9 Ko .js:19, .mjs:17, .scss:1
vite               29      73.5 Ko .js:18, .vue:5, .html:2, (sans extension):1
(racine)           23      11.3 Ko .js:12, (sans extension):7, .cjs:1, .json:1
scripts             4       4.2 Ko .sh:4

Fichiers a la racine :
  .c8rc
  .eslintignore
  .gitignore
  .gitmodules
  .npmignore
  .nycrc
  client.globe.js
  client.js
  client.map.js
  core.api.js
  core.client.js
  core.common.js
  LICENSE
  map.api.js
  map.client.globe.js
  map.client.js
  map.client.map.js
  map.common.js
  map.config.cjs
  package.json
  README.md
  sona

## 5. Classification des fichiers pour le RAG

Cette classification se place au niveau du depot complet. Elle sert a distinguer :
- ce qui doit etre indexe directement,
- ce qui doit etre indexe avec une strategie specialisee,
- ce qui peut rester secondaire ou etre exclu des premiers essais.


In [28]:
EXTENSIONS_RAG = {
    ".js", ".mjs", ".cjs", ".vue", ".ts", ".jsx", ".tsx",
    ".md", ".mdx", ".rst", ".txt",
    ".pdf", ".doc", ".docx", ".xls", ".xlsx", ".ppt", ".pptx", ".odt", ".ods",
    ".json", ".yaml", ".yml", ".toml",
    ".css", ".scss", ".sass", ".less", ".styl",
    ".html", ".ejs", ".pug",
}

fichiers_rag = [f for f in tous_fichiers if f.suffix.lower() in EXTENSIONS_RAG]
fichiers_autres = [f for f in tous_fichiers if f.suffix.lower() not in EXTENSIONS_RAG]

print(f"Fichiers pertinents pour le RAG : {len(fichiers_rag)}")
print(f"Fichiers non classes            : {len(fichiers_autres)}")

rag_par_ext = Counter(f.suffix.lower() for f in fichiers_rag)
print("\nRepartition des fichiers RAG :")
for ext, nb in rag_par_ext.most_common():
    taille = sum(f.stat().st_size for f in fichiers_rag if f.suffix.lower() == ext)
    print(f"  {ext:<10} {nb:>6} fichiers  {formater_taille(taille):>10}")

if fichiers_autres:
    print("\nExtensions non classees :")
    autres_ext = Counter(f.suffix.lower() if f.suffix else f.name for f in fichiers_autres)
    for ext, nb in autres_ext.most_common():
        print(f"  {ext:<20} {nb:>6}")



Fichiers pertinents pour le RAG : 810
Fichiers non classes            : 43

Repartition des fichiers RAG :
  .js           362 fichiers      1.9 Mo
  .vue          242 fichiers    832.5 Ko
  .md           141 fichiers    503.3 Ko
  .json          26 fichiers      3.5 Mo
  .mjs           18 fichiers     55.4 Ko
  .ejs            9 fichiers      4.3 Ko
  .cjs            6 fichiers     13.2 Ko
  .scss           2 fichiers      2.3 Ko
  .html           2 fichiers      3.7 Ko
  .css            1 fichiers     105.0 o
  .txt            1 fichiers     71.3 Ko

Extensions non classees :
  .xml                     16
  .drawio                   8
  .sh                       4
  .dods                     3
  .gitignore                2
  .c8rc                     1
  .eslintignore             1
  .gitmodules               1
  .npmignore                1
  .nycrc                    1
  LICENSE                   1
  .properties               1
  .skip                     1
  .das                   

## 6. Distribution par zone et par type de contenu

Cette analyse repond a une question structurante pour le chunking :
le depot est-il majoritairement documentaire, majoritairement source, ou mixte selon les zones ?


In [29]:
for zone in sorted(par_zone.keys()):
    fichiers = par_zone[zone]
    ext_counter = Counter(f.suffix.lower() if f.suffix else "(sans extension)" for f in fichiers)
    print(f"[{zone}]")
    for ext, nb in ext_counter.most_common(10):
        print(f"  {ext:<16} {nb:>4}")
    print()



[(racine)]
  .js                12
  (sans extension)    7
  .cjs                1
  .json               1
  .md                 1
  .properties         1

[core]
  .vue              150
  .js               149
  .json               6

[docs]
  .md               139
  .xml               15
  .drawio             8
  .json               2
  .mjs                1
  .css                1
  .js                 1

[extras]
  .js                19
  .mjs               17
  .scss               1

[map]
  .js               142
  .vue               87
  .json               6
  .cjs                3

[scripts]
  .sh                 4

[test]
  .js                21
  .json              10
  .ejs                9
  .dods               3
  .cjs                2
  .skip               1
  .txt                1
  .das                1
  .dds                1
  .xml                1

[vite]
  .js                18
  .vue                5
  .html               2
  (sans extension)    1
  .json          

## 7. Premiers signaux pour le chunking du code source

Avant de lancer un notebook specialise sur le code, on peut deja mesurer quelques signaux simples :
- nombre de fichiers de code par zone,
- densite en exports, fonctions et classes,
- presence de fichiers potentiellement tres denses semantiquement.

Ces signaux permettent de verifier si un simple chunking "par fichier" serait raisonnable.


In [30]:
CODE_EXTENSIONS = {".js", ".mjs", ".cjs", ".vue"}

def lire_texte(path):
    return path.read_text(encoding="utf-8", errors="ignore")

zones_code = ["core", "map", "vite", "test", "extras"]
stats_code = []
for zone in zones_code:
    fichiers = [f for f in par_zone.get(zone, []) if f.suffix.lower() in CODE_EXTENSIONS]
    nb_lignes = 0
    nb_exports = 0
    nb_fonctions = 0
    nb_classes = 0
    for f in fichiers:
        contenu = lire_texte(f)
        nb_lignes += len(contenu.splitlines())
        nb_exports += len(re.findall(r'^export\s', contenu, re.MULTILINE))
        nb_fonctions += len(re.findall(r'^(?:export\s+)?(?:async\s+)?function\s+', contenu, re.MULTILINE))
        nb_classes += len(re.findall(r'^(?:export\s+)?class\s+', contenu, re.MULTILINE))
    stats_code.append({
        "zone": zone,
        "fichiers": len(fichiers),
        "lignes": nb_lignes,
        "exports": nb_exports,
        "fonctions": nb_fonctions,
        "classes": nb_classes,
    })

print(f"{'Zone':<10} {'Fichiers':>8} {'Lignes':>8} {'Exports':>8} {'Fonctions':>10} {'Classes':>8}")
print("-" * 64)
for row in stats_code:
    print(f"{row['zone']:<10} {row['fichiers']:>8} {row['lignes']:>8} {row['exports']:>8} {row['fonctions']:>10} {row['classes']:>8}")



Zone       Fichiers   Lignes  Exports  Fonctions  Classes
----------------------------------------------------------------
core            299    29149      503        514        7
map             232    35108      512        530       16
vite             23     1680        9         11        0
test             23     4818        4          2        0
extras           36     2943      121        103        3


In [31]:
# Le score ci-dessous ne mesure pas la qualite du contenu,
# mais le nombre de points d'entree semantiques visibles dans un fichier.
densite_semantique = []
for zone in ["core", "map"]:
    for f in par_zone.get(zone, []):
        if f.suffix.lower() not in {".js", ".mjs", ".cjs"}:
            continue
        contenu = lire_texte(f)
        exports = len(re.findall(r'^export\s', contenu, re.MULTILINE))
        fonctions = len(re.findall(r'^(?:export\s+)?(?:async\s+)?function\s+', contenu, re.MULTILINE))
        classes = len(re.findall(r'^(?:export\s+)?class\s+', contenu, re.MULTILINE))
        score = exports + fonctions + classes
        densite_semantique.append({
            "fichier": str(f.relative_to(DATA_DIR)),
            "lignes": len(contenu.splitlines()),
            "exports": exports,
            "fonctions": fonctions,
            "classes": classes,
            "score": score,
        })

print("Fichiers source les plus denses semantiquement :")
for row in sorted(densite_semantique, key=lambda x: x["score"], reverse=True)[:20]:
    print(
        f"  {row['fichier']:<45} score={row['score']:>3} "
        f"exports={row['exports']:>2} fonctions={row['fonctions']:>2} "
        f"classes={row['classes']:>2} lignes={row['lignes']}"
    )



Fichiers source les plus denses semantiquement :
  map\client\utils\utils.layers.js              score= 77 exports=31 fonctions=46 classes= 0 lignes=713
  map\client\utils\utils.features.js            score= 61 exports=30 fonctions=31 classes= 0 lignes=642
  core\client\utils\index.js                    score= 47 exports=33 fonctions=14 classes= 0 lignes=218
  map\client\utils\utils.style.js               score= 43 exports=28 fonctions=15 classes= 0 lignes=436
  core\api\hooks\hooks.model.js                 score= 34 exports=17 fonctions=17 classes= 0 lignes=294
  map\api\services\index.js                     score= 33 exports=17 fonctions=16 classes= 0 lignes=310
  core\client\index.js                          score= 32 exports=32 fonctions= 0 classes= 0 lignes=109
  core\common\permissions.js                    score= 31 exports=17 fonctions=14 classes= 0 lignes=219
  map\common\wcs-utils.js                       score= 30 exports=15 fonctions=15 classes= 0 lignes=166
  map\common\op

## 8. Typologie des fichiers a traiter differemment dans le pipeline

Cette etape ne fixe pas encore les chunks, mais elle clarifie les grandes familles de traitement.


In [32]:
def categorie_fichier(path):
    rel = path.relative_to(DATA_DIR)
    parts = rel.parts
    ext = path.suffix.lower()

    if parts[0] == "docs" and ext == ".md":
        return "documentation_markdown"
    if ext == ".vue":
        return "vue_sfc"
    if path.name == "index.js":
        return "barrel_index_js"
    if parts[0] == "test":
        return "tests"
    if ext in {".js", ".mjs", ".cjs"}:
        return "code_js"
    if ext in {".json", ".yml", ".yaml", ".toml"}:
        return "configuration"
    if ext in {".md", ".txt"}:
        return "texte_hors_docs"
    if ext in {".html", ".ejs"}:
        return "template"
    return "autre"

par_categorie = Counter(categorie_fichier(f) for f in fichiers_rag)
print("Categories de traitement candidates :")
for categorie, nb in par_categorie.most_common():
    print(f"  {categorie:<24} {nb:>4}")



Categories de traitement candidates :
  code_js                   320
  vue_sfc                   242
  documentation_markdown    139
  barrel_index_js            46
  tests                      40
  configuration              16
  autre                       3
  texte_hors_docs             2
  template                    2


## 9. Resume et recommandations

`nb01` doit permettre de fixer le perimetre global du depot. L'analyse specialisee de `docs/`
reste dans `nb02`, mais on peut deja en deduire quelles familles de contenu auront besoin
de strategies de chunking differentes.


In [33]:
print("=" * 72)
print("RESUME - Vue generale du depot KDK")
print("=" * 72)
print(f"""
SCAN INITIAL :
  Fichiers trouves (hors repertoires exclus) : {len(tous_fichiers_bruts)}
  Repertoires exclus                         : {', '.join(sorted(EXCLUDED_DIRS))}

EXCLUSIONS :
  Fichiers exclus                            : {', '.join(sorted(EXCLUDED_FILES))}
  Extensions exclues                         : {', '.join(sorted(EXCLUDED_EXTENSIONS))}
  Total exclu                                : {nb_exclus} fichiers ({formater_taille(taille_exclue)})

RESULTAT GLOBAL :
  Fichiers retenus                           : {len(tous_fichiers)}
  Fichiers pertinents RAG                    : {len(fichiers_rag)}
  Fichiers non classes                       : {len(fichiers_autres)}
  Zones principales                          : {', '.join(sorted(par_zone.keys()))}

POINTS IMPORTANTS POUR LA SUITE :
  - le depot est majoritairement compose de code source (`core`, `map`)
  - `docs/` constitue une sous-partie structurante mais non representative du depot entier
  - plusieurs familles de contenu devront etre chunked differemment : Markdown, JS, Vue, tests, configs
  - la question du chunking du code peut etre preparee ici, mais devra etre analysee en detail dans un notebook dedie
""")

print("RECOMMANDATIONS POUR LA SUITE IMMEDIATE :")
print("  1. Conserver `nb01` comme vue d'ensemble du depot")
print("  2. Consolider `nb02` comme notebook de reference pour `docs/`")
print("  3. Ne pas extrapoler la strategie `docs/` a tout le depot sans etude du code source")
print("  4. Utiliser les categories ci-dessus pour definir plusieurs chunkers plutot qu'une regle unique")



RESUME - Vue generale du depot KDK

SCAN INITIAL :
  Fichiers trouves (hors repertoires exclus) : 970
  Repertoires exclus                         : .c8, .cache, .git, .github, .nyc_output, .output, __pycache__, build, coverage, dist, node_modules

EXCLUSIONS :
  Fichiers exclus                            : CHANGELOG.md, yarn.lock
  Extensions exclues                         : .gif, .lock, .png, .svg, .tif
  Total exclu                                : 117 fichiers (21.8 Mo)

RESULTAT GLOBAL :
  Fichiers retenus                           : 853
  Fichiers pertinents RAG                    : 810
  Fichiers non classes                       : 43
  Zones principales                          : (racine), core, docs, extras, map, scripts, test, vite

POINTS IMPORTANTS POUR LA SUITE :
  - le depot est majoritairement compose de code source (`core`, `map`)
  - `docs/` constitue une sous-partie structurante mais non representative du depot entier
  - plusieurs familles de contenu devront etre ch